# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mr-PeterMaged/flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

This playbook is built on the exact validated model from w05/w06, not a fresh refit: RF
trained on the 23 train clients, precision@50 = 0.78 and ROC-AUC = 0.604 measured on the 10
held-out test clients (GroupShuffleSplit, `random_state=42`). **The queue below covers only
those 10 held-out clients** -- they're the only pages this specific model has an honest,
out-of-sample precision@K for (more on why in section 2).

Each page gets **two independently-validated signals**, not one opaque score:
1. `decline_probability` -- the RF's probability of `is_declining_label`, from w05/w06.
2. `low_ctr_flag` -- the frozen ML-07 rule (CTR below its position tier's peer median).

Combining them (rather than picking one) gives four archetypes, each with one reason code and
one action label -- readable, and each half independently checked in an earlier week.

In [1]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

pd.set_option("display.width", 160)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DECISION_DATE = pd.Timestamp("2026-03-15")
RANDOM_SEED = 42

# identical query to w05/w06, plus dim_content.cpc for the cost/value view in section 1
df = con.execute(f"""
    WITH base AS (SELECT * FROM '{MARCH}' WHERE gsc_data_available IS TRUE),
    first_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_first_half,
               SUM(gsc_clicks) AS clicks_first_half,
               AVG(gsc_avg_position) AS avg_position_first_half,
               SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END)
                   AS ga4_engaged_sessions_first_half,
               COUNT(DISTINCT report_date) AS days_active_first_half
        FROM base
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
        GROUP BY client_hash_id, content_hash_id
    ),
    second_half AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_second_half
        FROM base
        WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT f.*, COALESCE(s.clicks_second_half, 0) AS clicks_second_half,
           dc.content_created_date, dc.cpc
    FROM first_half f
    LEFT JOIN second_half s USING (client_hash_id, content_hash_id)
    JOIN '{DIM_CONTENT}' dc USING (client_hash_id, content_hash_id)
    WHERE f.clicks_first_half > 0
      AND dc.is_published IS TRUE AND dc.is_deleted IS FALSE
      AND dc.content_created_date <= DATE '2026-03-15'
""").df()

df["is_declining_label"] = (df["clicks_second_half"] < df["clicks_first_half"]).astype(int)
df["ctr_first_half"] = df["clicks_first_half"] / df["impressions_first_half"]
df["content_age_days"] = (DECISION_DATE - pd.to_datetime(df["content_created_date"])).dt.days

def pos_tier(p):
    if p <= 0: return "no_rank"
    if p <= 3: return "1-3"
    if p <= 10: return "4-10"
    if p <= 20: return "11-20"
    return "21+"
df["position_tier"] = df["avg_position_first_half"].apply(pos_tier)

universe = df[(df["avg_position_first_half"] > 0) & (df["avg_position_first_half"] <= 20)
              & (df["impressions_first_half"] >= 200)].copy()
universe = universe.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

FEATURES = ["impressions_first_half", "clicks_first_half", "ctr_first_half",
            "avg_position_first_half", "ga4_engaged_sessions_first_half",
            "days_active_first_half", "content_age_days"]

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

RF_KWARGS = dict(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=RANDOM_SEED, n_jobs=-1)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(universe, groups=universe["client_hash_id"]))
train, test = universe.iloc[train_idx].copy(), universe.iloc[test_idx].copy()

rf = RandomForestClassifier(**RF_KWARGS).fit(train[FEATURES].fillna(0), train["is_declining_label"])
test = test.copy()
test["decline_probability"] = rf.predict_proba(test[FEATURES].fillna(0))[:, 1]

# frozen ML-07 baseline rule (peer-median CTR by tier) -- second, independent signal
PEER_MEDIAN_CTR = {"1-3": 0.002783, "4-10": 0.002101, "11-20": 0.001860}
test["peer_median_ctr"] = test["position_tier"].map(PEER_MEDIAN_CTR)
test["low_ctr_flag"] = test["ctr_first_half"] < test["peer_median_ctr"]

HIGH_RISK = 0.5
def archetype(row):
    if row["decline_probability"] >= HIGH_RISK and row["low_ctr_flag"]:
        return "decline_risk_and_ctr_gap", "priority_refresh_and_ctr_review"
    if row["decline_probability"] >= HIGH_RISK:
        return "predicted_decline_risk", "refresh_review"
    if row["low_ctr_flag"]:
        return "ctr_gap_only", "ctr_review"
    return "stable", "monitor"

codes = test.apply(archetype, axis=1)
test["reason_code"] = codes.apply(lambda t: t[0])
test["action_label"] = codes.apply(lambda t: t[1])

test["value_at_risk_usd"] = test["decline_probability"] * test["clicks_first_half"] * test["cpc"].fillna(0)

queue = test.sort_values(["decline_probability", "content_hash_id"], ascending=[False, True]).reset_index(drop=True)
queue["final_rank"] = queue.index + 1

print(f"queue: {len(queue):,} rows, {queue['client_hash_id'].nunique()} clients "
      f"(the 10 held-out test clients from w05/w06)")
print(f"\narchetype distribution:")
print(queue["reason_code"].value_counts())
print(f"\ntotal value at risk: ${queue['value_at_risk_usd'].sum():,.2f} "
      f"({(queue['cpc'] > 0).sum():,}/{len(queue):,} rows carry a nonzero CPC)")

cols = ["final_rank", "client_hash_id", "position_tier", "decline_probability",
        "low_ctr_flag", "reason_code", "action_label", "value_at_risk_usd"]
print("\ntop 10 by decline_probability (the primary queue order):")
print(queue[cols].head(10).to_string(index=False))

queue: 17,803 rows, 10 clients (the 10 held-out test clients from w05/w06)

archetype distribution:
reason_code
predicted_decline_risk      10945
ctr_gap_only                 4146
decline_risk_and_ctr_gap     1590
stable                       1122
Name: count, dtype: int64

total value at risk: $36,736.87 (4,537/17,803 rows carry a nonzero CPC)

top 10 by decline_probability (the primary queue order):
 final_rank          client_hash_id position_tier  decline_probability  low_ctr_flag            reason_code   action_label  value_at_risk_usd
          1 client_73cda7b4e4f265ea         11-20             0.704039         False predicted_decline_risk refresh_review           0.760362
          2 client_23a62021009f63c4          4-10             0.703952         False predicted_decline_risk refresh_review           0.000000
          3 client_fef1a8f436438636          4-10             0.703226         False predicted_decline_risk refresh_review           0.000000
          4 client_23a62021

**Archetype -> action mapping:**

| reason_code | action_label | n | typical next step |
|---|---|---|---|
| `decline_risk_and_ctr_gap` | `priority_refresh_and_ctr_review` | 1,590 | both signals agree -- review content AND snippet/CTR together, highest priority |
| `predicted_decline_risk` | `refresh_review` | 10,945 | model flags likely decline; review content freshness/relevance first |
| `ctr_gap_only` | `ctr_review` | 4,146 | not predicted to decline, but under-clicking for its position -- snippet/title review |
| `stable` | `monitor` | 1,122 | neither signal fires -- no action needed this cycle |

**The decay/refresh insight -- and where it narrows the paper's finding.** The FlyRank paper's
Finding #2 reads content age as a portfolio-wide decay driver (health score falls from 33.1 at
61-90 days to 14.0 at 271-365 days). This lane's own evidence points somewhere more specific:
permutation importance in w05/w06 showed `content_age_days` contributing ~0 to this model's
ranking, and ML-07's bucket check found no clean CTR-vs-age relationship either -- checked
twice, two different ways, both negative. **Content age is deliberately left out of both the
score and the archetype rule here.** The two findings aren't in conflict: the paper measures a
composite health score across an entire 57-brand portfolio; this lane measures one narrow
proxy (does clicks drop from one half of a month to the next) on 10 clients. The honest
reading is that age may matter for the paper's broader outcome without mattering for this
specific one -- not that one of the two checks is wrong.

**Cost/value thinking.** `value_at_risk_usd` (`decline_probability x clicks_first_half x cpc`)
follows the paper's own Finding #9 rule -- clicks x CPC, never impressions x CPC. It is shown
as a **secondary sort a reviewer can apply, not folded into the primary ranking**: 75% of rows
carry `cpc = 0` (no assigned commercial value), and the single highest value-at-risk page in
this queue (\$1,401.85 -- 165 clicks x \$14.68 CPC x 0.58 probability) sits at `decline_probability
= 0.579`, well down the probability-only list. A reviewer working a value-constrained calendar
should re-sort by `value_at_risk_usd`; a reviewer working pure content-health should keep the
default `decline_probability` order. Neither order is "more correct" -- they answer different
questions.

## 2. Intended use and limits

**Intended use:** a Lane 2 content reviewer with a fixed weekly review budget, deciding which
pages to look at first across one client's inventory (or a small set of clients), for the
March 2026 snapshot of the FlyRank internship warehouse.

**Where this stops being valid:**
- **Client scope.** This queue covers only the **10 held-out test clients** the model was
  validated on. The other 23 clients in the warehouse were used to *train* this model --
  scoring their own pages with it would be in-sample and optimistically biased, so they are
  deliberately excluded here rather than shown with an unearned confidence number.
- **Time scope.** Everything is built from a single mid-panel month (March 2026), with the
  label itself a same-month proxy (first-half clicks vs. second-half clicks), not a true
  prior-90-day -> next-30-day outcome. April, or any other month, is untested.
- **What the probability means.** precision@50 = 0.78 means roughly 39 of the top 50 flagged
  pages carried the label in this one test split -- a **directional, decision-support**
  number (per `writing-honest-claims`), not a guarantee for any individual page, and not
  evidence that *acting* on a flag improves anything (this is an observational proxy label,
  not an experiment).
- **What's excluded on purpose.** GA4 coverage is thin (4.2% of March rows, from ML-04) --
  `ga4_engaged_sessions_first_half` is close to a constant zero for most pages and carries
  little independent signal. `content_age_days` is computed but not used in the score
  (section 1). No client names, URLs, or raw queries appear anywhere below.

In [2]:
print(f"validated clients in this queue: {queue['client_hash_id'].nunique()} "
      f"(of {universe['client_hash_id'].nunique()} total in the March 2026 universe)")
print(f"validated period: 2026-03-01 to 2026-03-31 (feature window 03-01..15, label window 03-16..31)")
print(f"validated performance (w05/w06, same split): precision@50 = 0.78, ROC-AUC = 0.604, "
      f"test base rate = {queue['is_declining_label'].mean():.1%}")
print(f"GA4 rows with real signal in this queue: {(queue['ga4_engaged_sessions_first_half'] > 0).sum()} "
      f"/ {len(queue)} ({(queue['ga4_engaged_sessions_first_half'] > 0).mean():.1%})")

validated clients in this queue: 10 (of 33 total in the March 2026 universe)
validated period: 2026-03-01 to 2026-03-31 (feature window 03-01..15, label window 03-16..31)
validated performance (w05/w06, same split): precision@50 = 0.78, ROC-AUC = 0.604, test base rate = 53.7%
GA4 rows with real signal in this queue: 2217 / 17803 (12.5%)


## 3. Human review + the no-go list

**A person must check, before acting on any flagged page:**
1. **Near-zero-click, high-impression rows are often tracking artifacts, not content
   problems** -- w05's own error analysis found exactly this pattern (thousands of
   impressions, 0-1 clicks). Check GA4/GSC coverage for the URL before writing new copy.
2. **Client concentration** -- ML-07 found 6 of its top 10 belonged to one client, because a
   raw/absolute score favors high-traffic clients. Check below whether this queue has the same
   skew before treating "top of the list" as "most urgent across clients."
3. **Business context the model can't see** -- is the page already scheduled for a redesign,
   deprecation, or a campaign that explains a dip? The model only sees clicks and position.
4. **Duplicate effort** -- check whether a page was already reviewed/refreshed this cycle
   before assigning it again.

**What should NOT be automated, ever, from this notebook's output:**
- No auto-publishing, auto-editing, or auto-generating page titles/meta/content from a score.
- No auto-unpublishing, de-indexing, or deleting `stable`/`monitor` pages -- absence of a flag
  is not evidence a page is safe to remove.
- No treating one page's `decline_probability` as a certainty -- it is a portfolio ranking
  tool; at precision@50 = 0.78, roughly 1 in 5 of even the top 50 is wrong.
- No using this queue, or any client's position in it, as a performance-review or staffing
  input for a content team.
- No extending the queue to a client outside the validated 10 and presenting its scores with
  the same confidence as the validated ones.

In [3]:
top20_clients = queue.head(20)["client_hash_id"].value_counts()
print("client_hash_id counts within this queue's top 20 (check for concentration):")
print(top20_clients)

near_zero = queue[(queue["clicks_first_half"] >= 20) & (queue["clicks_first_half"] <= 200) &
                   (queue["decline_probability"] >= HIGH_RISK)]
print(f"\nflagged rows worth a tracking-artifact sanity check "
      f"(moderate first-half clicks, flagged high-risk): {len(near_zero)}")

client_hash_id counts within this queue's top 20 (check for concentration):
client_hash_id
client_23a62021009f63c4    13
client_fef1a8f436438636     3
client_9958f0a7ae1df715     2
client_73cda7b4e4f265ea     1
client_a80fca3f171ed1de     1
Name: count, dtype: int64

flagged rows worth a tracking-artifact sanity check (moderate first-half clicks, flagged high-risk): 1888


**Finding: this queue has the same concentration pattern ML-07 flagged.** 13 of the top 20
rows belong to a single client (`client_23a62021...`); the next-largest contributor supplies
only 3. A raw probability ranking, like a raw score ranking, still favors whichever client
happens to have the most borderline-declining pages this month. **A reviewer working across
multiple clients should cap how many consecutive slots one client can take, or review
within-client, rather than trusting one global top-20 as "the 20 most urgent pages
company-wide."** This is the same lesson from ML-07, now confirmed on a different ranking
method -- it is a property of ranking a multi-client portfolio globally, not a quirk of one
particular score.

## 4. Monitoring / retrain triggers

What would tell a future run this playbook has gone stale, checked against the **stored
reference values from this run**:

| Signal | Reference value (this run) | Trigger |
|---|---|---|
| precision@50 on a fresh held-out month | 0.78 | recompute on the new month's held-out clients; investigate if it drops below ~0.65 (still beats the 0.54 baseline/dummy floor, but a real drop) |
| Label base rate (test split) | 53.7% | recompute monthly; investigate if it moves outside roughly 40-65% -- a sign the underlying dynamic (or the label definition) shifted |
| GSC/GA4 availability share | 36.7% / 4.2% (ML-04) | recompute monthly; a large drop means fewer rows have real signal to rank on |
| Top permutation-importance feature | `ctr_first_half` (w05/w06) | if a different feature takes over the top spot, the signal itself may have shifted -- re-run the leakage/signal audit before trusting the new ranking |
| Client set | 10 held-out / 33 total | any client not in the original 33 is **unvalidated by definition** -- retrain-and-reholdout before scoring them, don't just apply the existing model |

**Retrain cadence:** monthly, always on a fresh **mid-panel** month (never the sealed final
month of whatever warehouse snapshot is current -- same rule as ML-04/ML-07), with a new
`GroupShuffleSplit` and a fresh precision@K/ROC-AUC readout before the new queue replaces this
one.

In [4]:
reference_values = {
    "precision_at_50": 0.78,
    "roc_auc": 0.604,
    "test_base_rate": round(float(queue["is_declining_label"].mean()), 3),
    "gsc_availability_share_ml04": 0.367,
    "ga4_availability_share_ml04": 0.042,
    "top_permutation_importance_feature": "ctr_first_half",
    "validated_client_count": int(queue["client_hash_id"].nunique()),
    "total_client_count": int(universe["client_hash_id"].nunique()),
}
for k, v in reference_values.items():
    print(f"{k}: {v}")

precision_at_50: 0.78
roc_auc: 0.604
test_base_rate: 0.537
gsc_availability_share_ml04: 0.367
ga4_availability_share_ml04: 0.042
top_permutation_importance_feature: ctr_first_half
validated_client_count: 10
total_client_count: 33


## 5. Exports for the paper

The full queue (all columns, all 10 validated clients) goes to `work/outputs/` -- it stays out
of git by design (the CI leak-guard blocks data files; this cell regenerates it on every run).
The metrics JSON and two figures are the committed receipts the paper builds on.

In [5]:
import matplotlib.pyplot as plt
import json as _json

OUT_DIR = "../outputs"
FIG_DIR = "../figures"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

export_cols = ["final_rank", "client_hash_id", "content_hash_id", "position_tier",
               "impressions_first_half", "clicks_first_half", "ctr_first_half",
               "avg_position_first_half", "decline_probability", "low_ctr_flag",
               "reason_code", "action_label", "value_at_risk_usd", "is_declining_label"]
queue_path = f"{OUT_DIR}/w07_action_queue.csv"
queue[export_cols].to_csv(queue_path, index=False)
print(f"wrote {queue_path} ({len(queue):,} rows)")

metrics = {
    "assignment": "ML-10 (w07_action_playbook.ipynb)",
    "lane": "Lane 2 -- Refresh / Content Opportunity Scoring",
    "queue_scope": {
        "validated_clients": int(queue["client_hash_id"].nunique()),
        "total_clients_in_universe": int(universe["client_hash_id"].nunique()),
        "queue_rows": int(len(queue)),
    },
    "validated_performance_w05_w06": {"precision_at_50": 0.78, "roc_auc": 0.604,
                                       "test_base_rate": round(float(queue["is_declining_label"].mean()), 3)},
    "archetype_distribution": queue["reason_code"].value_counts().to_dict(),
    "value_at_risk_usd": {
        "total": round(float(queue["value_at_risk_usd"].sum()), 2),
        "rows_with_nonzero_cpc": int((queue["cpc"] > 0).sum()),
        "top_row_usd": round(float(queue["value_at_risk_usd"].max()), 2),
    },
    "monitoring_reference_values": reference_values,
}
metrics_path = "../outputs/w07_playbook_metrics.json"
with open(metrics_path, "w") as f:
    _json.dump(metrics, f, indent=2)
print(f"wrote {metrics_path}")

# figure 1: precision@50 comparison (frozen numbers from w05/w06 -- same test split)
fig, ax = plt.subplots(figsize=(6, 4))
methods = ["Dummy", "Baseline rule\n(ML-07)", "Logistic Regression\n(scaled)", "Random Forest"]
values = [0.54, 0.54, 0.58, 0.78]
bars = ax.bar(methods, values, color=["#9aa5b1", "#9aa5b1", "#6b8caf", "#2e5c8a"])
ax.set_ylabel("precision@50")
ax.set_title("precision@50 by method (held-out test clients, w05/w06)")
ax.set_ylim(0, 1)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/precision_at_50_comparison.png", dpi=150)
plt.close(fig)
print(f"wrote {FIG_DIR}/precision_at_50_comparison.png")

# figure 2: archetype distribution in this queue
fig, ax = plt.subplots(figsize=(7, 4))
dist = queue["reason_code"].value_counts()
ax.barh(dist.index[::-1], dist.values[::-1], color="#2e5c8a")
ax.set_xlabel("pages")
ax.set_title("Action archetype distribution\n(10 validated clients)")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/archetype_distribution.png", dpi=150)
plt.close(fig)
print(f"wrote {FIG_DIR}/archetype_distribution.png")

wrote ../outputs/w07_action_queue.csv (17,803 rows)
wrote ../outputs/w07_playbook_metrics.json


wrote ../figures/precision_at_50_comparison.png
wrote ../figures/archetype_distribution.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled -- markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.